# Project 2.2 — Random Walk
**Topic:** Lattice random walks and the emergence of diffusion. Connecting microscopic randomness to the macroscopic diffusion equation.


---
## 1. Set Up

### Physical Motivation
A **random walk** is the microscopic model for **Brownian motion**: a particle takes unit steps in randomly chosen directions at each time step. The key result is that the mean-squared displacement grows linearly in time:

$$\langle r^2(t) \rangle = 2d\,D\,t$$

where $d$ is the spatial dimension and $D$ is the diffusion coefficient. For a lattice walk with step size $\ell$ and time step $\tau$:

$$D = \frac{\ell^2}{2d\,\tau}$$

This is **Einstein's relation** (1905). The random walk is the discrete precursor to the diffusion equation $\partial_t \rho = D\nabla^2\rho$.

### What "2D Lattice Walk" Means
At each step, one coordinate axis is chosen at random, and the particle steps $\pm 1$ along that axis. This is the simplest unbiased lattice walk: no preferred direction, no memory (Markov property).

### Key Observable — $\langle r^2 \rangle$ vs $t$
The slope of $\langle r^2(t)\rangle$ vs. $t$ on a linear plot gives $2dD$. The code fits this slope and extracts $D$. Deviations from linearity at early times are **transient effects** (the walk hasn't "forgotten" its starting point yet); deviations at late times are **finite-sample effects** (fewer walkers survive to large $r$).

### ⚠️ Limitations
1. **Lattice artefacts**: a lattice walk has discrete step sizes, so $P(r,t)$ is non-Gaussian at short times. Convergence to Gaussian is only in the $N_{\text{steps}} \to \infty$ limit (CLT).
2. **Finite ensemble**: $\langle r^2 \rangle$ estimated from $M$ walkers has variance $\sim r^4/M$. For $M = 6000$, relative error is $\sim 1/\sqrt{6000} \approx 1.3\%$.
3. **Isotropy on square lattice**: the 2-D lattice walk is isotropic on average but not at short times — the choice of axis introduces a weak anisotropy that washes out at large $N_{\text{steps}}$.


In [ ]:
import os, shutil
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

OUT = Path("output_random_walk")
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(20260526)
np.set_printoptions(precision=6, suppress=True)
print(f"Output directory: {OUT.resolve()}")


---
## 2. Functions

### `simulate_random_walk_statistics()`

```python
positions = np.zeros((n_walkers, dimension), dtype=np.int64)
```
Stores the $(x, y, \ldots)$ coordinates of all walkers simultaneously. Integer positions are exact for a lattice walk — no floating-point accumulation error. For $M = 6000$ walkers in 2-D: $6000 \times 2 \times 8 = 96\,\text{KB}$ — fits comfortably in L2 cache.

```python
axes = rng.integers(0, dimension, size=n_walkers)
signs = rng.choice(np.array([-1, 1], dtype=np.int64), size=n_walkers)
positions[np.arange(n_walkers), axes] += signs
```
Vectorised step for all walkers at once. **Key design decision**: instead of a Python loop over walkers (slow), we use NumPy index tricks:
- `axes` selects which coordinate to update (0=$x$, 1=$y$).
- `signs` selects $+1$ or $-1$.
- `positions[np.arange(n_walkers), axes]` — advanced indexing: selects element `axes[i]` from row `i` for all `i` simultaneously.

This is the critical computational bottleneck: the loop is over **time steps**, not walkers. For $N_{\text{steps}} = 2000$ and $M = 6000$, this saves a factor of $6000$ compared to a naive double loop.

```python
r2 = np.sum(positions * positions, axis=1)
mean_r2[step] = r2.mean()
```
Computes $r^2 = x^2 + y^2 + \ldots$ for all walkers, then averages. `axis=1` sums over coordinates, keeping one $r^2$ per walker.

### `fit_diffusion_slope()`

```python
coeff = np.polyfit(steps[start:], mean_r2[start:], deg=1)
```
Fits $\langle r^2 \rangle = 2dD \cdot n$ (linear in step number $n$). **Why skip the first `fit_start_fraction`?** At early steps, $\langle r^2\rangle$ has a transient regime before asymptotic diffusive behaviour sets in. Skipping 20% of the data avoids biasing the fit.

**Theoretical slope**: for 2-D, slope $= 2d \cdot D = 2 \times 2 \times \frac{1}{2\times 2} = 1$ (step size = 1, $d = 2$). For 3-D, slope $= 1$ as well with this normalisation. This is a useful check.

### 🔧 Improvement Directions
1. **Self-avoiding walk (SAW)**: forbid revisiting lattice sites. The SAW has anomalous diffusion: $\langle r^2\rangle \sim n^{2\nu}$ with $\nu \approx 0.75$ in 2-D. This is relevant to polymer physics.
2. **Biased walk**: add a drift $p_+ = 0.5 + \epsilon$. Show that $\langle r^2\rangle$ still grows linearly but $\langle r\rangle \sim v_{\text{drift}} t$ — drift plus diffusion.
3. **Histogram evolution**: plot $P(x,t)$ at several times — it should converge to a Gaussian with variance $\sigma^2 = 2Dt$.
4. **Lévy flight**: replace uniform step size with a power-law distribution. Get anomalous (super)diffusion.

### ⚠️ Known Weaknesses
- The `sample` trajectory storage scales as $N_{\text{steps}} \times n_{\text{sample}} \times d$ — for large runs, this dominates memory.
- `np.arange(n_walkers)` creates a temporary array every step — for very large $M$, create it once outside the loop.


In [ ]:
def simulate_random_walk_statistics(n_steps=2000, n_walkers=6000,
                                    dimension=2, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    positions = np.zeros((n_walkers, dimension), dtype=np.int64)
    mean_r2   = np.empty(n_steps + 1, dtype=float)
    mean_r    = np.empty(n_steps + 1, dtype=float)
    mean_r2[0] = 0.0
    mean_r[0]  = 0.0

    n_sample = min(6, n_walkers)
    sample = np.zeros((n_steps + 1, n_sample, dimension), dtype=np.int64)

    idx_all = np.arange(n_walkers)              # pre-compute to avoid re-allocation
    for step in range(1, n_steps + 1):
        axes  = rng.integers(0, dimension, size=n_walkers)    # which axis to step
        signs = rng.choice(np.array([-1, 1], dtype=np.int64), size=n_walkers)
        positions[idx_all, axes] += signs        # vectorised step

        r2 = np.sum(positions * positions, axis=1)   # r^2 for each walker
        mean_r2[step] = r2.mean()
        mean_r[step]  = np.sqrt(r2).mean()
        sample[step]  = positions[:n_sample]

    steps = np.arange(n_steps + 1)
    return steps, mean_r2, mean_r, sample

def fit_diffusion_slope(steps, mean_r2, fit_start_fraction=0.2):
    start = int(fit_start_fraction * len(steps))
    coeff = np.polyfit(steps[start:], mean_r2[start:], deg=1)
    return coeff[0], coeff[1]    # slope = 2*d*D, intercept


---
## 3 & 4. Calculation & Writeouts

### Extracting the Diffusion Coefficient
From the fit slope $s = 2dD$:

$$D = \frac{s}{2d}$$

For a lattice walk with unit step and unit time: $D_{\text{theory}} = \ell^2/(2d\tau) = 1/(2d)$.
- 2-D: $D_{\text{theory}} = 1/4$, slope = $2 \times 2 \times 1/4 = 1.0$
- 3-D: $D_{\text{theory}} = 1/6$, slope = $2 \times 3 \times 1/6 = 1.0$

Both have **the same slope** — a coincidence of this normalisation. The diffusion coefficient itself differs.


In [ ]:
N_STEPS   = 2000
N_WALKERS = 6000

results_rw = {}
for dim in (2, 3):
    steps, mean_r2, mean_r, sample = simulate_random_walk_statistics(
        n_steps=N_STEPS, n_walkers=N_WALKERS, dimension=dim, rng=rng)
    slope, intercept = fit_diffusion_slope(steps, mean_r2)
    D_fit    = slope / (2 * dim)
    D_theory = 1.0 / (2 * dim)
    results_rw[dim] = dict(steps=steps, mean_r2=mean_r2, mean_r=mean_r,
                           sample=sample, slope=slope, D_fit=D_fit,
                           D_theory=D_theory)
    print(f"{dim}-D Random Walk:")
    print(f"  Fitted slope  = {slope:.5f}  (theory: 1.0000)")
    print(f"  D_fit         = {D_fit:.5f}  (theory: {D_theory:.5f})")
    print(f"  Relative error: {abs(D_fit-D_theory)/D_theory*100:.2f}%")
    print()


---
## 5. Matplotlib Visualisation

### What to Look For
- **$\langle r^2\rangle$ vs. step**: linear after the initial transient. The fitted line should pass through the data well. Any systematic curvature suggests either drift (break of isotropy) or correlations in the RNG.
- **Sample trajectories**: these illustrate the qualitative difference between 2-D and 3-D walks — in 2-D the walk is **recurrent** (returns to origin with probability 1), in 3-D it is **transient** (escapes to infinity with positive probability). This is Pólya's theorem (1921).


In [ ]:
fig = plt.figure(figsize=(15, 10))

for i, dim in enumerate([2, 3]):
    r = results_rw[dim]
    steps, mean_r2, sample = r['steps'], r['mean_r2'], r['sample']
    slope, D_fit = r['slope'], r['D_fit']

    # MSD plot
    ax1 = fig.add_subplot(2, 3, i*3 + 1)
    ax1.plot(steps, mean_r2, 'steelblue', lw=1.5, label='$\langle r^2 \rangle$')
    fit_line = slope * steps + 0
    ax1.plot(steps, fit_line, 'r--', lw=1.5,
             label=f'Fit slope={slope:.3f}, D={D_fit:.4f}')
    ax1.set_xlabel('Step $n$')
    ax1.set_ylabel('$\langle r^2 \rangle$')
    ax1.set_title(f'{dim}-D: Mean Squared Displacement')
    ax1.legend(fontsize=9)

    # Log-log plot to check power law
    ax2 = fig.add_subplot(2, 3, i*3 + 2)
    steps_nz = steps[1:]
    ax2.loglog(steps_nz, mean_r2[1:], 'steelblue', lw=1.5)
    ax2.loglog(steps_nz, steps_nz, 'r--', lw=1.2, label='Slope 1 (diffusion)')
    ax2.set_xlabel('$\log(n)$')
    ax2.set_ylabel('$\log(\langle r^2\rangle)$')
    ax2.set_title(f'{dim}-D: Log-log (slope=1 = normal diffusion)')
    ax2.legend(fontsize=9)

    # Sample trajectories
    ax3 = fig.add_subplot(2, 3, i*3 + 3,
                          projection='3d' if dim == 3 else None)
    colors_traj = plt.cm.tab10(np.linspace(0, 0.6, sample.shape[1]))
    for j, c in zip(range(sample.shape[1]), colors_traj):
        traj = sample[:, j, :]
        if dim == 2:
            ax3.plot(traj[:, 0], traj[:, 1], color=c, lw=0.7, alpha=0.7)
            ax3.plot(0, 0, 'ko', ms=4)
        else:
            ax3.plot(traj[:, 0], traj[:, 1], traj[:, 2], color=c, lw=0.5, alpha=0.7)
    ax3.set_title(f'{dim}-D Sample Trajectories')

plt.tight_layout()
plt.savefig(OUT / "random_walk.png", dpi=150, bbox_inches='tight')
plt.show()


---
## 6. Sanity Check & Validation

| Test | Criterion | Physical reason |
|------|-----------|-----------------|
| $\langle r^2(0) \rangle = 0$ | All walkers start at origin | Deterministic initial condition |
| $\langle r^2\rangle$ monotone | Non-decreasing (on average) | Diffusion is irreversible |
| Slope $\approx 1.0$ | $< 2\%$ error | Einstein relation |
| Gaussian at large $N$ | K-S test on final positions | CLT |


In [ ]:
from scipy import stats

print("=" * 55)
print("SANITY CHECKS — Project 02.2 Random Walk")
print("=" * 55)

for dim, r in results_rw.items():
    steps, mean_r2 = r['steps'], r['mean_r2']
    print(f"\n── {dim}-D ──")

    # 1. Initial condition
    ok = abs(mean_r2[0]) < 0.01
    print(f"  1. <r²(0)> = {mean_r2[0]:.4f}  (expected 0)  {'✓' if ok else '✗'}")

    # 2. Slope accuracy
    rel_err = abs(r['slope'] - 1.0)
    ok = rel_err < 0.05
    print(f"  2. Slope = {r['slope']:.5f}  (expected 1.0)  {'✓' if ok else '✗'}")

    # 3. Diffusion coefficient
    rel_err_D = abs(r['D_fit'] - r['D_theory']) / r['D_theory']
    ok = rel_err_D < 0.05
    print(f"  3. D_fit = {r['D_fit']:.5f}, D_theory = {r['D_theory']:.5f}  "
          f"err={rel_err_D*100:.2f}%  {'✓' if ok else '✗'}")

    # 4. Monotonicity (mean should not decrease significantly)
    diffs = np.diff(mean_r2)
    large_drops = np.sum(diffs < -0.5)
    ok = large_drops == 0
    print(f"  4. Large drops in <r²>: {large_drops}  {'✓' if ok else '✗'}")

    # 5. Gaussian test on final-step x-positions across walkers
    _, sample_data, _ = simulate_random_walk_statistics(
        n_steps=500, n_walkers=2000, dimension=dim, rng=np.random.default_rng(42))
    final_x = sample_data[-1, :, 0] if dim >= 2 else sample_data[-1, :, 0]

print("\n" + "=" * 55)
